## 1. Initialize Project Environment
Import libraries for data manipulation, normalization, and filtering.

In [10]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from scipy import stats

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)

pandas 2.2.3
numpy 2.1.3


## 2. Define Configuration Parameters
Centralize paths, filtering thresholds, and normalization options for reproducibility.

In [11]:
@dataclass
class PrepConfig:
    handle: str = "AndreiCod"
    # GSE50081 NSCLC (Non-Small Cell Lung Cancer) data - contains TP53!
    # Dataset: 181 tumor samples, gene-level expression, Affymetrix GPL570
    geo_data_file: Path = Path(
        "../../../data/work/AndreiCod/lab06/GSE50081_gene_expression.csv"
    )
    geo_metadata_file: Path = Path(
        "../../../data/work/AndreiCod/lab06/GSE50081_metadata.csv"
    )
    # Target gene for correlation analysis
    target_gene: str = "TP53"
    # Filtering parameters for bulk RNA-seq
    top_variable_genes: int = 2000  # Select top N most variable genes
    variance_threshold: float = 0.3  # Variance threshold
    export_dir: Path = Path("artifacts")
    seed: int = 42

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["geo_data_file"] = str(info["geo_data_file"])
        info["geo_metadata_file"] = str(info["geo_metadata_file"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = PrepConfig()
CONFIG.describe()

{'handle': 'AndreiCod',
 'geo_data_file': '../../../data/work/AndreiCod/lab06/GSE50081_gene_expression.csv',
 'geo_metadata_file': '../../../data/work/AndreiCod/lab06/GSE50081_metadata.csv',
 'target_gene': 'TP53',
 'top_variable_genes': 2000,
 'variance_threshold': 0.3,
 'export_dir': 'artifacts',
 'seed': 42}

## 3. Load GSE50081 Non-Small Cell Lung Cancer RNA-Seq Data
Load the expression data from GEO (GSE50081: Der et al., Clin Cancer Res 2014).
This dataset contains bulk RNA-seq from 181 NSCLC tumor samples with clinical annotations.
**Importantly, this cancer dataset contains TP53 expression** - a key tumor suppressor often 
mutated/dysregulated in lung cancer, matching the assignment requirement for TP53-associated analysis.

In [12]:
def load_geo_expression_data(cfg: PrepConfig) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load GSE50081 NSCLC bulk RNA-seq expression data and sample metadata.

    Returns:
        expr_df: Expression matrix (genes x samples)
        metadata: Sample metadata with clinical info
    """
    logging.info(f"Loading GSE50081 expression data...")

    # Load expression matrix (genes as rows, samples as columns)
    expr_df = pd.read_csv(cfg.geo_data_file, index_col=0)
    logging.info(f"  Raw data: {expr_df.shape[0]} genes x {expr_df.shape[1]} samples")

    # Check for target gene (TP53)
    if cfg.target_gene in expr_df.index:
        tp53_expr = expr_df.loc[cfg.target_gene]
        logging.info(f"\n*** TARGET GENE {cfg.target_gene} FOUND! ***")
        logging.info(
            f"  Expression range: {tp53_expr.min():.2f} - {tp53_expr.max():.2f}"
        )
        logging.info(f"  Mean expression: {tp53_expr.mean():.2f}")
    else:
        logging.warning(f"Target gene {cfg.target_gene} not found in dataset!")

    # Load sample metadata
    logging.info(f"\nLoading sample metadata...")
    metadata = pd.read_csv(cfg.geo_metadata_file)
    logging.info(f"  Samples: {len(metadata)}")
    logging.info(f"  Metadata columns: {list(metadata.columns)}")

    # Show clinical distribution
    if "histology" in metadata.columns:
        logging.info("\nHistology distribution:")
        for hist, count in metadata["histology"].value_counts().items():
            logging.info(f"  {hist}: {count} samples")

    if "Stage" in metadata.columns:
        logging.info("\nCancer stage distribution:")
        for stage, count in metadata["Stage"].value_counts().items():
            logging.info(f"  {stage}: {count} samples")

    return expr_df, metadata


expr_raw, metadata = load_geo_expression_data(CONFIG)
print(f"\nExpression matrix shape: {expr_raw.shape}")
print(f"\nSample info:")
print(metadata[["SampleID", "histology", "Stage", "status"]].head(10))

[INFO] Loading GSE50081 expression data...
[INFO]   Raw data: 2001 genes x 181 samples
[INFO] 
*** TARGET GENE TP53 FOUND! ***
[INFO]   Expression range: 3.81 - 7.78
[INFO]   Mean expression: 5.44
[INFO] 
Loading sample metadata...
[INFO]   Samples: 181
[INFO]   Metadata columns: ['SampleID', 'Title', 'Sex', 'histology', 't-stage', 'n-stage', 'm-stage', 'Stage', 'age', 'smoking', 'survival_time', 'status', 'disease-free_survival_time', 'recurrence']
[INFO] 
Histology distribution:
[INFO]   adenocarcinoma: 127 samples
[INFO]   squamous cell carcinoma: 42 samples
[INFO]   large cell carcinoma: 7 samples
[INFO]   adenosquamous carcinoma: 2 samples
[INFO]   NSClarge cell carcinoma-mixed: 1 samples
[INFO]   NSCLC-favor adenocarcinoma: 1 samples
[INFO]   squamous cell carcinoma X2: 1 samples
[INFO] 
Cancer stage distribution:
[INFO]   1B: 79 samples
[INFO]   1A: 48 samples
[INFO]   2B: 45 samples
[INFO]   2A: 9 samples



Expression matrix shape: (2001, 181)

Sample info:
     SampleID                histology Stage status
0  GSM1213669  adenosquamous carcinoma    1A   dead
1  GSM1213670           adenocarcinoma    1A  alive
2  GSM1213671           adenocarcinoma    1B  alive
3  GSM1213672           adenocarcinoma    1A  alive
4  GSM1213673           adenocarcinoma    1B   dead
5  GSM1213674  squamous cell carcinoma    1B  alive
6  GSM1213675  squamous cell carcinoma    2B  alive
7  GSM1213676           adenocarcinoma    1B   dead
8  GSM1213677           adenocarcinoma    1B  alive
9  GSM1213678           adenocarcinoma    2B   dead


In [13]:
# View clinical outcome distribution
print("Status distribution (survival outcome):")
print(metadata["status"].value_counts())
print("\nRecurrence distribution:")
print(metadata["recurrence"].value_counts())

Status distribution (survival outcome):
status
alive    106
dead      75
Name: count, dtype: int64

Recurrence distribution:
recurrence
N    126
Y     51
U      4
Name: count, dtype: int64


## 4. Preprocessing: Variance Filtering and Gene Selection
For bulk RNA-seq microarray data (Affymetrix, already normalized), we:
1. Data is already log-transformed (Affymetrix RMA normalized)
2. Select top variable genes for network construction
3. **Ensure TP53 is included** for downstream analysis

In [14]:
def select_top_variable_genes(
    df: pd.DataFrame, n_top: int, ensure_gene: str = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Select top N most variable genes based on variance across samples.
    Optionally ensures a specific gene is included.
    """
    variances = df.var(axis=1)
    variance_df = pd.DataFrame({"Gene": df.index, "Variance": variances}).sort_values(
        "Variance", ascending=False
    )

    top_genes = variance_df.head(n_top)["Gene"].tolist()

    # Ensure target gene is included
    if ensure_gene and ensure_gene in df.index and ensure_gene not in top_genes:
        top_genes.append(ensure_gene)
        logging.info(
            f"Added {ensure_gene} to gene set (not in top {n_top} by variance)"
        )

    df_top = df.loc[top_genes]

    logging.info(f"Variable gene selection (top {n_top}):")
    logging.info(f"  Mean variance: {variances.mean():.4f}")
    logging.info(f"  Top gene variance: {variance_df['Variance'].iloc[0]:.4f}")
    logging.info(
        f"  Min selected variance: {variance_df['Variance'].iloc[min(n_top - 1, len(variance_df) - 1)]:.4f}"
    )

    if ensure_gene:
        if ensure_gene in df_top.index:
            gene_var = variances[ensure_gene]
            gene_rank = (variances > gene_var).sum() + 1
            logging.info(f"  {ensure_gene} variance: {gene_var:.4f} (rank {gene_rank})")
        else:
            logging.warning(f"  {ensure_gene} NOT in final gene set!")

    return df_top, variance_df


def filter_low_variance(
    df: pd.DataFrame, threshold: float, ensure_gene: str = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Filter genes with variance below threshold, optionally keeping a specific gene."""
    variances = df.var(axis=1)
    variance_df = pd.DataFrame({"Gene": df.index, "Variance": variances}).sort_values(
        "Variance", ascending=False
    )

    mask = variances >= threshold

    # Always keep the target gene
    if ensure_gene and ensure_gene in df.index:
        mask[ensure_gene] = True

    df_filtered = df[mask]

    logging.info(f"Variance filtering (threshold={threshold}):")
    logging.info(f"  Genes before: {df.shape[0]}")
    logging.info(f"  Genes after: {df_filtered.shape[0]}")
    logging.info(f"  Removed: {df.shape[0] - df_filtered.shape[0]}")

    return df_filtered, variance_df


# Apply preprocessing pipeline
print("=" * 60)
print("PREPROCESSING PIPELINE - GSE50081 NSCLC Data")
print("=" * 60)

# The data is already RMA-normalized (log2 scale) from Affymetrix
expr_log = expr_raw.copy()
logging.info(f"Data already normalized (Affymetrix RMA)")
logging.info(
    f"  Value range: {expr_log.values.min():.2f} - {expr_log.values.max():.2f}"
)

# Select top variable genes (ensuring TP53 is included)
expr_top, variance_stats = select_top_variable_genes(
    expr_log, CONFIG.top_variable_genes, ensure_gene=CONFIG.target_gene
)

# Final variance filter (keeping TP53)
expr_filtered, _ = filter_low_variance(
    expr_top, CONFIG.variance_threshold, ensure_gene=CONFIG.target_gene
)

print(f"\nFinal preprocessed matrix shape: {expr_filtered.shape}")
print(f"TP53 in final gene set: {'TP53' in expr_filtered.index}")

[INFO] Data already normalized (Affymetrix RMA)
[INFO]   Value range: 1.92 - 14.53
[INFO] Added TP53 to gene set (not in top 2000 by variance)
[INFO] Variable gene selection (top 2000):
[INFO]   Mean variance: 1.6704
[INFO]   Top gene variance: 15.5904
[INFO]   Min selected variance: 0.7994
[INFO]   TP53 variance: 0.4240 (rank 2001)
[INFO] Variance filtering (threshold=0.3):
[INFO]   Genes before: 2001
[INFO]   Genes after: 2001
[INFO]   Removed: 0


PREPROCESSING PIPELINE - GSE50081 NSCLC Data

Final preprocessed matrix shape: (2001, 181)
TP53 in final gene set: True


In [15]:
# Variance statistics - top 20 genes
print("Top 20 highest variance genes:")
variance_stats.head(20)

Top 20 highest variance genes:


,Gene,Variance
Gene,,
KRT6A,KRT6A,15.590383
SPRR1B,SPRR1B,10.886695
SCGB1A1,SCGB1A1,10.199984
RPS4Y1,RPS4Y1,10.007082
SPINK1,SPINK1,9.947358
KRT5,KRT5,9.807973
AKR1B10,AKR1B10,9.746627
SCGB3A2,SCGB3A2,9.128093
SFTA2,SFTA2,9.008272


In [16]:
# Summary statistics of preprocessed data
summary_stats = {
    "raw_genes": expr_raw.shape[0],
    "raw_samples": expr_raw.shape[1],
    "top_variable_genes_selected": CONFIG.top_variable_genes,
    "final_genes": expr_filtered.shape[0],
    "variance_threshold": CONFIG.variance_threshold,
    "mean_expression": expr_filtered.values.mean(),
    "std_expression": expr_filtered.values.std(),
    "target_gene": CONFIG.target_gene,
    "target_gene_included": CONFIG.target_gene in expr_filtered.index,
}
pd.DataFrame([summary_stats]).T.rename(columns={0: "Value"})

,Value
raw_genes,2001
raw_samples,181
top_variable_genes_selected,2000
final_genes,2001
variance_threshold,0.3
mean_expression,6.260014
std_expression,2.203539
target_gene,TP53
target_gene_included,True


## 5. Validate with Unit Tests
Sanity checks for the preprocessing pipeline.

In [17]:
def test_variance_filter():
    test_df = pd.DataFrame(
        {"S1": [1, 10, 100], "S2": [1, 20, 200], "S3": [1, 30, 300]},
        index=["low_var", "med_var", "high_var"],
    )
    filtered, _ = filter_low_variance(test_df, threshold=50)
    assert "high_var" in filtered.index


def test_expression_shape():
    """Verify preprocessed data has expected structure."""
    # Allow +1 for TP53 if not in top N
    assert expr_filtered.shape[0] <= CONFIG.top_variable_genes + 1, (
        "More genes than expected"
    )
    assert expr_filtered.shape[1] == expr_raw.shape[1], "Sample count changed"


def test_tp53_included():
    """Verify target gene TP53 is in final dataset."""
    assert CONFIG.target_gene in expr_filtered.index, (
        f"{CONFIG.target_gene} missing from filtered data!"
    )


test_variance_filter()
test_expression_shape()
test_tp53_included()
print("All preprocessing tests passed.")
print(f"✓ TP53 confirmed in final gene set!")

[INFO] Variance filtering (threshold=50):
[INFO]   Genes before: 3
[INFO]   Genes after: 2
[INFO]   Removed: 1


All preprocessing tests passed.
✓ TP53 confirmed in final gene set!


## 6. Export Results
Save preprocessed expression matrix and metadata for downstream tasks.

In [18]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save preprocessed expression matrix
expr_filtered.to_csv(EXPORT_DIR / "task1_expression_preprocessed.csv")
print(
    f"[OK] Preprocessed expression saved to: {EXPORT_DIR / 'task1_expression_preprocessed.csv'}"
)

# Save raw expression subset (top genes only, to reduce file size)
expr_log.loc[expr_filtered.index].to_csv(EXPORT_DIR / "task1_expression_raw.csv")
print(
    f"[OK] Raw expression (top genes) saved to: {EXPORT_DIR / 'task1_expression_raw.csv'}"
)

# Save sample metadata with clinical info
metadata.to_csv(EXPORT_DIR / "task1_sample_metadata.csv", index=False)
print(f"[OK] Sample metadata saved to: {EXPORT_DIR / 'task1_sample_metadata.csv'}")

# Save variance statistics
variance_stats.to_csv(EXPORT_DIR / "task1_variance_stats.csv", index=False)
print(f"[OK] Variance statistics saved to: {EXPORT_DIR / 'task1_variance_stats.csv'}")

# Save TP53 expression profile for correlation analysis
tp53_profile = expr_filtered.loc[CONFIG.target_gene]
tp53_profile.to_csv(EXPORT_DIR / "task1_tp53_expression.csv", header=True)
print(
    f"[OK] TP53 expression profile saved to: {EXPORT_DIR / 'task1_tp53_expression.csv'}"
)

print(f"\n[OK] All Task 1 artifacts exported to: {EXPORT_DIR}")
print(f"\nDataset Summary:")
print(f"  - GSE50081 Non-Small Cell Lung Cancer")
print(f"  - {expr_filtered.shape[0]} genes x {expr_filtered.shape[1]} tumor samples")
print(f"  - TP53 included: Yes")
print(f"  - Ready for WGCNA co-expression network construction")

[OK] Preprocessed expression saved to: artifacts/task1_expression_preprocessed.csv
[OK] Raw expression (top genes) saved to: artifacts/task1_expression_raw.csv
[OK] Sample metadata saved to: artifacts/task1_sample_metadata.csv
[OK] Variance statistics saved to: artifacts/task1_variance_stats.csv
[OK] TP53 expression profile saved to: artifacts/task1_tp53_expression.csv

[OK] All Task 1 artifacts exported to: artifacts

Dataset Summary:
  - GSE50081 Non-Small Cell Lung Cancer
  - 2001 genes x 181 tumor samples
  - TP53 included: Yes
  - Ready for WGCNA co-expression network construction
